# 08 · Singly Linked List

A **singly linked list** is a chain of **nodes**. Each node stores a `val` and a pointer `next` to the following node. The last node's `next` is `None`.

```
head → [5 | •] → [2 | •] → [3 | •] → [4 | None]
```

- We can only move **forward** (never backward).
- No random access: reaching the i-th node takes O(i) steps.
- Inserting/deleting at the front is O(1) (just re-link pointers), unlike a Python list.

**Topics covered**
1. Building nodes by hand
2. A `SinglyLinkedList` class: append, traverse, insert, delete
3. Middle of the list (counting vs. slow/fast pointers)
4. Reverse a linked list
5. Cycle detection (hash set vs. Floyd), cycle start, cycle length
6. Odd-even list
7. Remove the N-th node from the end

## 1. Creating nodes manually

> 🐞 **Fix:** the earlier version wrote `node1.next = node(2)`, which creates a *brand-new* node instead of linking the existing `node2`. Link the objects you already created.

In [1]:
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None

node1, node2, node3, node4 = Node(5), Node(2), Node(3), Node(4)
node1.next = node2          # 5 → 2
node2.next = node3          # 2 → 3
node3.next = node4          # 3 → 4

print(node1.val)            # 5
print(node1.next.val)       # 2
print(node1.next.next.val)  # 3

5
2
3


## 2. `SinglyLinkedList` class

| Method | Idea | Time |
|---|---|---|
| `append(val)` | walk to the last node, link the new node | O(n) |
| `traverse()` | walk from `head` until `None` | O(n) |
| `insert_at(val, pos)` | stop at the node **before** `pos`, re-link two pointers | O(n) |
| `delete(val)` | find the node, make the **previous** node skip it | O(n) |

**Inserting between `prev` and `curr`:**
```
new.next = curr
prev.next = new
```
(Always set `new.next` first, otherwise you lose the rest of the list.)

> 🐞 **Fixes vs. the earlier version**
> - `delete` removed the *last* node no matter which value you asked for, and did nothing if the list had one element. Rewritten properly.
> - `traverse` printed values with no separator; it now prints `5 → 2 → 3`.
> - `insert_at` now reports an out-of-range position instead of crashing.

In [2]:
class SinglyLinkedList:
    def __init__(self):
        self.head = None

    def append(self, val):
        new_node = Node(val)
        if self.head is None:                  # empty list: new node is the head
            self.head = new_node
            return
        curr = self.head
        while curr.next is not None:           # go to the LAST node
            curr = curr.next
        curr.next = new_node

    def traverse(self):
        if self.head is None:
            print("SLL is empty")
            return
        values = []
        curr = self.head
        while curr is not None:
            values.append(str(curr.val))
            curr = curr.next
        print(" → ".join(values))

    def insert_at(self, val, position):
        new_node = Node(val)
        if position == 0:                      # insert at head
            new_node.next = self.head
            self.head = new_node
            return
        prev = self.head
        for _ in range(position - 1):          # move to the node just before `position`
            if prev is None:
                break
            prev = prev.next
        if prev is None:
            print("Position out of bounds")
            return
        new_node.next = prev.next              # 1) new node points to the old next
        prev.next = new_node                   # 2) previous node points to new node

    def delete(self, val):
        if self.head is None:
            print("SLL is empty")
            return
        if self.head.val == val:               # deleting the head
            self.head = self.head.next
            return
        prev, curr = self.head, self.head.next
        while curr is not None:
            if curr.val == val:
                prev.next = curr.next          # skip over the node
                return
            prev, curr = curr, curr.next
        print("Node not found")

sll = SinglyLinkedList()
for v in [10, 20, 30]:
    sll.append(v)
sll.traverse()                 # 10 → 20 → 30
sll.insert_at(5, 0)
sll.insert_at(15, 2)
sll.traverse()                 # 5 → 10 → 15 → 20 → 30
sll.delete(15)
sll.delete(5)
sll.delete(30)
sll.traverse()                 # 10 → 20
sll.delete(99)                 # Node not found

10 → 20 → 30
5 → 10 → 15 → 20 → 30
10 → 20
Node not found


## Helpers used in the rest of this notebook
`build_list` creates a list from Python values, `to_list` converts back so results are easy to print and test.

In [3]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def to_list(head):
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out

print(to_list(build_list([1, 2, 3])))

[1, 2, 3]


## 3. Middle of the linked list

For an even length, return the **second** of the two middle nodes (`[1..6]` → starts at `4`).

**Approach 1 – count first:** pass 1 counts `n`; pass 2 moves `n // 2` steps. Two passes, O(n).

**Approach 2 – slow & fast pointers (tortoise & hare):** `slow` moves 1 step, `fast` moves 2. When `fast` reaches the end, `slow` is in the middle. One pass, O(n).

In [4]:
def middle_node_count(head):
    n, temp = 0, head
    while temp:
        n += 1
        temp = temp.next
    temp = head
    for _ in range(n // 2):
        temp = temp.next
    return temp

def middle_node(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next            # 1 step
        fast = fast.next.next       # 2 steps
    return slow

for values in ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]):
    head = build_list(values)
    print(values, "→", to_list(middle_node_count(head)), to_list(middle_node(head)))

[1, 2, 3, 4, 5] → [3, 4, 5] [3, 4, 5]
[1, 2, 3, 4, 5, 6] → [4, 5, 6] [4, 5, 6]


## 4. Reverse a linked list

Use **three pointers** and flip each `next` arrow as we walk:

- `prev` = already-reversed part (starts as `None`)
- `curr` = node being processed
- `front` = saved next node (so we don't lose the rest)

```
None ← 1   2 → 3 → 4          step: save front=2, point 1.next to None, move prev=1, curr=2
```

**Time:** O(n) · **Space:** O(1)

In [5]:
def reverse_list(head):
    prev = None
    curr = head
    while curr is not None:
        front = curr.next        # 1) remember the rest of the list
        curr.next = prev         # 2) flip the arrow
        prev = curr              # 3) advance prev
        curr = front             # 4) advance curr
    return prev                  # prev is the new head

print(to_list(reverse_list(build_list([1, 2, 3, 4, 5]))))   # [5, 4, 3, 2, 1]

# Recursive version: reverse the rest, then make the next node point back to us
def reverse_list_recursive(head):
    if head is None or head.next is None:
        return head
    new_head = reverse_list_recursive(head.next)
    head.next.next = head        # the node after me now points to me
    head.next = None
    return new_head

print(to_list(reverse_list_recursive(build_list([1, 2, 3, 4, 5]))))

[5, 4, 3, 2, 1]
[5, 4, 3, 2, 1]


## 5. Cycle detection

A cycle exists when some node's `next` points back to an earlier node, so walking never reaches `None`.

**Approach 1 – hash set:** store visited *nodes*; if we see one again there is a cycle. O(n) time, O(n) space.

**Approach 2 – Floyd's slow/fast pointers:** if there is a cycle, `fast` eventually laps `slow` and they meet. O(n) time, **O(1)** space.

> 🐞 **Fix:** the hash version previously called `my_set()` (a set isn't callable) and returned the node instead of a boolean.

In [6]:
def has_cycle_hash(head):
    seen = set()
    temp = head
    while temp is not None:
        if temp in seen:             # same node object visited twice
            return True
        seen.add(temp)
        temp = temp.next
    return False

def has_cycle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:             # they met inside the cycle
            return True
    return False

# Build 1 → 2 → 3 → 4 → 5 and optionally close a loop
def make_cyclic(values, cycle_to_index):
    nodes = [ListNode(v) for v in values]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if cycle_to_index is not None:
        nodes[-1].next = nodes[cycle_to_index]
    return nodes[0]

print(has_cycle(make_cyclic([1, 2, 3, 4, 5], None)), has_cycle_hash(make_cyclic([1, 2, 3, 4, 5], None)))  # False False
print(has_cycle(make_cyclic([1, 2, 3, 4, 5], 2)),    has_cycle_hash(make_cyclic([1, 2, 3, 4, 5], 2)))     # True True

False False
True True


### Where does the cycle start?
After `slow` and `fast` meet:
1. Move `slow` back to `head`.
2. Move **both** one step at a time.
3. The node where they meet again is the **start of the cycle**.

(Math: the distance from `head` to the cycle start equals the distance from the meeting point to the cycle start, going around.)

In [7]:
def detect_cycle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            slow = head                  # restart one pointer from the head
            while slow is not fast:
                slow = slow.next
                fast = fast.next
            return slow                  # cycle start
    return None

start = detect_cycle(make_cyclic([1, 2, 3, 4, 5], 2))
print("Cycle starts at node with value:", start.val)    # 3
print(detect_cycle(make_cyclic([1, 2, 3], None)))        # None

Cycle starts at node with value: 3
None


### Length of the cycle
When `slow` and `fast` meet inside the loop, keep moving **one** pointer until it returns to the meeting node, counting steps.

> 🐞 **Test fix:** the earlier test linked the last node back to `node2` (loop of 4 nodes: 2, 3, 4, 5) but expected `3`. The code was right; the test was wrong. To get a loop of 3 nodes, link back to `node3`.

In [8]:
def cycle_length(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            count = 1
            slow = slow.next
            while slow is not fast:
                slow = slow.next
                count += 1
            return count
    return 0                              # no cycle

print(cycle_length(make_cyclic([1, 2, 3, 4, 5], 2)))   # 3  (nodes 3, 4, 5)
print(cycle_length(make_cyclic([1, 2, 3, 4, 5], 1)))   # 4  (nodes 2, 3, 4, 5)
print(cycle_length(make_cyclic([1, 2, 3], None)))      # 0

3
4
0


## 6. Odd-even linked list

Group all nodes at **odd positions** first (1st, 3rd, 5th …) followed by nodes at **even positions** (2nd, 4th …). Keep the original relative order in each group.

`1 → 2 → 3 → 4 → 5`  becomes  `1 → 3 → 5 → 2 → 4`

**Idea:** build two chains by re-linking `next` pointers (odd chain and even chain) in one pass, remember the head of the even chain, and attach it to the end of the odd chain.

**Time:** O(n) · **Space:** O(1)

In [9]:
def odd_even_list(head):
    if head is None or head.next is None:
        return head
    odd = head
    even = head.next
    even_head = even                  # remember where the even chain starts
    while even is not None and even.next is not None:
        odd.next = even.next          # odd skips over the even node
        odd = odd.next
        even.next = odd.next          # even skips over the odd node
        even = even.next
    odd.next = even_head              # glue the chains together
    return head

print(to_list(odd_even_list(build_list([1, 2, 3, 4, 5]))))        # [1, 3, 5, 2, 4]
print(to_list(odd_even_list(build_list([1, 2, 3, 4, 5, 6]))))     # [1, 3, 5, 2, 4, 6]

[1, 3, 5, 2, 4]
[1, 3, 5, 2, 4, 6]


## 7. Remove the N-th node from the end

**Approach 1 – two passes:** count the length `L`; the node to delete is at index `L − n` from the front, so stop at the node before it. If `L == n`, we are removing the head.

**Approach 2 – one pass with two pointers:** move `fast` **n steps ahead**, then move `slow` and `fast` together. When `fast` reaches the last node, `slow` is right before the node to delete (the gap between them stays `n`).

In [10]:
def remove_nth_two_pass(head, n):
    length, temp = 0, head
    while temp:
        length += 1
        temp = temp.next
    if length == n:                       # removing the head
        return head.next
    curr = head
    for _ in range(length - n - 1):       # stop at the node BEFORE the target
        curr = curr.next
    curr.next = curr.next.next
    return head

def remove_nth_from_end(head, n):
    slow = fast = head
    for _ in range(n):                    # create a gap of n nodes
        fast = fast.next
    if fast is None:                      # gap reached past the end → delete the head
        return head.next
    while fast.next is not None:
        slow = slow.next
        fast = fast.next
    slow.next = slow.next.next            # unlink the target
    return head

print(to_list(remove_nth_two_pass(build_list([1, 2, 3, 4, 5]), 2)))    # [1, 2, 3, 5]
print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))    # [1, 2, 3, 5]
print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 5)))    # [2, 3, 4, 5] (head removed)
print(to_list(remove_nth_from_end(build_list([1]), 1)))                # []

[1, 2, 3, 5]
[1, 2, 3, 5]
[2, 3, 4, 5]
[]


---
### ✅ Key takeaways
- To **insert/delete**, you always need the node *before* the position.
- **Slow & fast pointers** solve: middle node, cycle detection, cycle start/length.
- To reverse, save `front = curr.next` *before* overwriting `curr.next`.
- A **dummy node** (used in `build_list`) removes many special cases for the head.
- "N-th from end" → create a gap of `n` between two pointers.